# 📊 Embryoscope Embryo Metrics: DuckDB vs AWS Athena

This notebook connects to both the local **DuckDB** database and **AWS Athena (Production)** to execute data quality queries on the `embryoscope_embrioes` table. It retrieves:
1. Total row count
2. Unique embryo IDs
3. Unique patient IDs
4. Unique patient IDxs
5. Unique `prontuario` (medical record numbers)
6. Percentage of patients/records with a valid `prontuario` (not null or -1)

Metrics are calculated overall, per year (`embryo_EmbryoDate`), and per clinic location.

In [16]:
import duckdb
import pandas as pd
from pyathena import connect

# Connections setup
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
print("Connecting to DuckDB...")
duck_con = duckdb.connect(DUCKDB_PATH, read_only=True)

print("Connecting to AWS Athena...")
ath_con = connect(
    region_name="sa-east-1",
    work_group="datalake-admins"
)
ath_cur = ath_con.cursor()

Connecting to DuckDB...
Connecting to AWS Athena...


## 🔍 Part 1: DuckDB Queries (Local Gold Layer)
Running local metrics on `gold.embryoscope_embrioes` including `patient_PatientIDx`.

In [17]:
# 1. DuckDB Overall
duck_overall_q = """
SELECT 
    'TOTAL' as grouping_type,
    'ALL' as grouping_value,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_EmbryoID) as unique_embryos,
    COUNT(DISTINCT patient_PatientID) as unique_patients,
    COUNT(DISTINCT patient_PatientIDx) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_PatientID END) * 100.0 / COUNT(DISTINCT patient_PatientID), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold.embryoscope_embrioes;
"""
df_duck_overall = duck_con.execute(duck_overall_q).df()
print("--- DuckDB Overall Totals ---")
display(df_duck_overall)

# 2. DuckDB Per Year
duck_year_q = """
SELECT 
    EXTRACT(YEAR FROM embryo_EmbryoDate) as embryo_year,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_EmbryoID) as unique_embryos,
    COUNT(DISTINCT patient_PatientID) as unique_patients,
    COUNT(DISTINCT patient_PatientIDx) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_PatientID END) * 100.0 / COUNT(DISTINCT patient_PatientID), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold.embryoscope_embrioes
GROUP BY 1
ORDER BY 1;
"""
df_duck_year = duck_con.execute(duck_year_q).df()
print("\n--- DuckDB Per Year ---")
display(df_duck_year)

# 3. DuckDB Per Location
duck_loc_q = """
SELECT 
    patient_unit_huntington,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_EmbryoID) as unique_embryos,
    COUNT(DISTINCT patient_PatientID) as unique_patients,
    COUNT(DISTINCT patient_PatientIDx) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_PatientID END) * 100.0 / COUNT(DISTINCT patient_PatientID), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold.embryoscope_embrioes
GROUP BY 1
ORDER BY 1;
"""
df_duck_loc = duck_con.execute(duck_loc_q).df()
print("\n--- DuckDB Per Location ---")
display(df_duck_loc)

--- DuckDB Overall Totals ---


,grouping_type,grouping_value,total_rows,unique_embryos,unique_patients,unique_patient_idxs,unique_prontuarios,pct_patients_with_valid_prontuario,pct_rows_with_valid_prontuario
0,TOTAL,ALL,148830,148830,13091,13704,12681,99.42,98.67



--- DuckDB Per Year ---


,embryo_year,total_rows,unique_embryos,unique_patients,unique_patient_idxs,unique_prontuarios,pct_patients_with_valid_prontuario,pct_rows_with_valid_prontuario
0,2017,602,602,81,81,81,100.00,100.00
1,2018,4127,4127,455,458,455,99.78,99.81
2,2019,14459,14459,1642,1671,1638,99.39,99.48
3,2020,17364,17364,1799,1824,1775,99.11,95.66
4,2021,21180,21180,2298,2346,2258,98.96,98.96
5,2022,19872,19872,2102,2147,2043,99.38,98.68
6,2023,19231,19231,1986,2009,1963,99.40,99.34
7,2024,17330,17330,1761,1777,1752,99.72,99.45
8,2025,22164,22164,2205,2242,2191,99.73,99.18
9,2026,12501,12501,1296,1309,1289,99.77,97.98



--- DuckDB Per Location ---


,patient_unit_huntington,total_rows,unique_embryos,unique_patients,unique_patient_idxs,unique_prontuarios,pct_patients_with_valid_prontuario,pct_rows_with_valid_prontuario
0,Belo Horizonte,26949,26949,2538,2538,2424,99.01,99.16
1,Brasilia,18532,18532,1783,1786,1738,98.43,94.68
2,Ibirapuera,65836,65836,5948,6070,5873,99.73,99.04
3,Vila Mariana,37513,37513,3299,3310,3261,99.64,99.66


## ☁️ Part 2: AWS Athena Queries (Production Gold Layer)
Running metrics on `gold_huntington_prod.embryoscope_embrioes` including `patient_id_x`.

In [18]:
# 1. Athena Overall
ath_overall_q = """
SELECT 
    'TOTAL' as grouping_type,
    'ALL' as grouping_value,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_embryo_id) as unique_embryos,
    COUNT(DISTINCT patient_id) as unique_patients,
    COUNT(DISTINCT patient_id_x) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_id END) * 100.0 / COUNT(DISTINCT patient_id), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold_huntington_prod.embryoscope_embrioes;
"""
ath_cur.execute(ath_overall_q)
df_ath_overall = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
print("--- Athena Overall Totals ---")
display(df_ath_overall)

# 2. Athena Per Year
ath_year_q = """
SELECT 
    EXTRACT(YEAR FROM embryo_embryo_date) as embryo_year,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_embryo_id) as unique_embryos,
    COUNT(DISTINCT patient_id) as unique_patients,
    COUNT(DISTINCT patient_id_x) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_id END) * 100.0 / COUNT(DISTINCT patient_id), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold_huntington_prod.embryoscope_embrioes
GROUP BY 1
ORDER BY 1;
"""
ath_cur.execute(ath_year_q)
df_ath_year = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
print("\n--- Athena Per Year ---")
display(df_ath_year)

# 3. Athena Per Location
ath_loc_q = """
SELECT 
    unit_huntington,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_embryo_id) as unique_embryos,
    COUNT(DISTINCT patient_id) as unique_patients,
    COUNT(DISTINCT patient_id_x) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_id END) * 100.0 / COUNT(DISTINCT patient_id), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold_huntington_prod.embryoscope_embrioes
GROUP BY 1
ORDER BY 1;
"""
ath_cur.execute(ath_loc_q)
df_ath_loc = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
print("\n--- Athena Per Location ---")
display(df_ath_loc)

--- Athena Overall Totals ---


,grouping_type,grouping_value,total_rows,unique_embryos,unique_patients,unique_patient_idxs,unique_prontuarios,pct_patients_with_valid_prontuario,pct_rows_with_valid_prontuario
0,TOTAL,ALL,143980,143980,12763,13047,4426,36.33,33.93



--- Athena Per Year ---


,embryo_year,total_rows,unique_embryos,unique_patients,unique_patient_idxs,unique_prontuarios,pct_patients_with_valid_prontuario,pct_rows_with_valid_prontuario
0,2017,602,602,81,81,5,4.94,2.82
1,2018,4175,4175,460,460,19,5.00,5.05
2,2019,14465,14465,1652,1672,328,20.58,18.19
3,2020,17391,17391,1809,1827,648,36.82,32.32
4,2021,21406,21406,2317,2350,933,41.09,39.62
5,2022,20511,20511,2138,2162,941,45.42,42.33
6,2023,19554,19554,2005,2022,660,33.37,31.26
7,2024,17883,17883,1773,1786,664,37.68,36.20
8,2025,15092,15092,1406,1410,633,45.31,45.26
9,2026,12901,12901,1316,1319,374,28.57,29.39



--- Athena Per Location ---


,unit_huntington,total_rows,unique_embryos,unique_patients,unique_patient_idxs,unique_prontuarios,pct_patients_with_valid_prontuario,pct_rows_with_valid_prontuario
0,Belo Horizonte,27000,27000,2542,2542,2425,98.98,99.09
1,Brasilia,19360,19360,1804,1804,1708,95.73,89.40
2,Ibirapuera,58973,58973,5386,5386,154,3.95,4.41
3,Vila Mariana,38647,38647,3315,3315,187,6.06,5.66


## ⚖️ Part 3: Comparative Breakdown (DuckDB vs Athena Prod)
Below, we merge the results to highlight discrepancies, now containing unique Patient IDx counts.

In [19]:
# 1. Location Comparison
df_duck_loc_clean = df_duck_loc.rename(columns={
    'patient_unit_huntington': 'location',
    'total_rows': 'duck_rows',
    'unique_patients': 'duck_patients',
    'unique_patient_idxs': 'duck_patient_idxs',
    'unique_prontuarios': 'duck_prontuarios',
    'pct_patients_with_valid_prontuario': 'duck_valid_pront_pct'
})[['location', 'duck_rows', 'duck_patients', 'duck_patient_idxs', 'duck_prontuarios', 'duck_valid_pront_pct']]

df_ath_loc_clean = df_ath_loc.rename(columns={
    'unit_huntington': 'location',
    'total_rows': 'ath_rows',
    'unique_patients': 'ath_patients',
    'unique_patient_idxs': 'ath_patient_idxs',
    'unique_prontuarios': 'ath_prontuarios',
    'pct_patients_with_valid_prontuario': 'ath_valid_pront_pct'
})[['location', 'ath_rows', 'ath_patients', 'ath_patient_idxs', 'ath_prontuarios', 'ath_valid_pront_pct']]

df_comp_loc = pd.merge(df_duck_loc_clean, df_ath_loc_clean, on='location')
print("--- Side-by-side Location Comparison ---")
display(df_comp_loc)

# 2. Year Comparison
df_duck_yr_clean = df_duck_year.rename(columns={
    'embryo_year': 'year',
    'total_rows': 'duck_rows',
    'unique_patients': 'duck_patients',
    'unique_patient_idxs': 'duck_patient_idxs',
    'unique_prontuarios': 'duck_prontuarios',
    'pct_patients_with_valid_prontuario': 'duck_valid_pront_pct'
})[['year', 'duck_rows', 'duck_patients', 'duck_patient_idxs', 'duck_prontuarios', 'duck_valid_pront_pct']]

df_ath_yr_clean = df_ath_year.rename(columns={
    'embryo_year': 'year',
    'total_rows': 'ath_rows',
    'unique_patients': 'ath_patients',
    'unique_patient_idxs': 'ath_patient_idxs',
    'unique_prontuarios': 'ath_prontuarios',
    'pct_patients_with_valid_prontuario': 'ath_valid_pront_pct'
})[['year', 'ath_rows', 'ath_patients', 'ath_patient_idxs', 'ath_prontuarios', 'ath_valid_pront_pct']]

# Ensure both are integers for seamless merge
df_duck_yr_clean['year'] = df_duck_yr_clean['year'].astype(float).astype(int)
df_ath_yr_clean['year'] = df_ath_yr_clean['year'].astype(float).astype(int)

df_comp_yr = pd.merge(df_duck_yr_clean, df_ath_yr_clean, on='year')
print("\n--- Side-by-side Year Comparison ---")
display(df_comp_yr)

--- Side-by-side Location Comparison ---


,location,duck_rows,duck_patients,duck_patient_idxs,duck_prontuarios,duck_valid_pront_pct,ath_rows,ath_patients,ath_patient_idxs,ath_prontuarios,ath_valid_pront_pct
0,Belo Horizonte,26949,2538,2538,2424,99.01,27000,2542,2542,2425,98.98
1,Brasilia,18532,1783,1786,1738,98.43,19360,1804,1804,1708,95.73
2,Ibirapuera,65836,5948,6070,5873,99.73,58973,5386,5386,154,3.95
3,Vila Mariana,37513,3299,3310,3261,99.64,38647,3315,3315,187,6.06



--- Side-by-side Year Comparison ---


,year,duck_rows,duck_patients,duck_patient_idxs,duck_prontuarios,duck_valid_pront_pct,ath_rows,ath_patients,ath_patient_idxs,ath_prontuarios,ath_valid_pront_pct
0,2017,602,81,81,81,100.00,602,81,81,5,4.94
1,2018,4127,455,458,455,99.78,4175,460,460,19,5.00
2,2019,14459,1642,1671,1638,99.39,14465,1652,1672,328,20.58
3,2020,17364,1799,1824,1775,99.11,17391,1809,1827,648,36.82
4,2021,21180,2298,2346,2258,98.96,21406,2317,2350,933,41.09
5,2022,19872,2102,2147,2043,99.38,20511,2138,2162,941,45.42
6,2023,19231,1986,2009,1963,99.40,19554,2005,2022,660,33.37
7,2024,17330,1761,1777,1752,99.72,17883,1773,1786,664,37.68
8,2025,22164,2205,2242,2191,99.73,15092,1406,1410,633,45.31
9,2026,12501,1296,1309,1289,99.77,12901,1316,1319,374,28.57


## 🔍 Part 4: Discrepant Patient Sample Comparison (First Record)
This section finds 5 patients who have a valid `prontuario` in DuckDB, but have a missing/default (`NULL` or `-1`) `prontuario` in AWS Athena. It retrieves and displays the first record (line) for each of these 5 patients from both sources for direct comparison.

In [20]:
# 1. Fetch unique patients (at patient_PatientIDx level) and their prontuarios
df_duck_pts = duck_con.execute("""
    SELECT DISTINCT patient_PatientIDx, patient_PatientID, patient_FirstName as first_name, prontuario
    FROM gold.embryoscope_embrioes
    WHERE prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1'
""").df()

ath_cur.execute("""
    SELECT DISTINCT patient_id_x, patient_id, first_name, prontuario
    FROM gold_huntington_prod.embryoscope_embrioes
""")
df_ath_pts = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])

# Convert IDxs and IDs to string to prevent merge type mismatch
df_duck_pts['patient_PatientIDx'] = df_duck_pts['patient_PatientIDx'].astype(str)
df_ath_pts['patient_id_x'] = df_ath_pts['patient_id_x'].astype(str)

# 2. Find discrepancies where Athena prontuario is NULL or -1, joining on patient_id_x
merged_pts = pd.merge(df_duck_pts, df_ath_pts, left_on='patient_PatientIDx', right_on='patient_id_x', suffixes=('_duck', '_ath'))
discrepant = merged_pts[
    merged_pts['prontuario_ath'].isna() | 
    (merged_pts['prontuario_ath'] == -1) | 
    (merged_pts['prontuario_ath'].astype(str) == '-1') | 
    (merged_pts['prontuario_ath'].astype(str) == 'None')
].drop_duplicates(subset=['patient_PatientIDx']).head(5)

sample_patient_idxs = discrepant['patient_PatientIDx'].tolist()
print("Discrepant Patient IDxs identified:", sample_patient_idxs)

# 3. Fetch all records for only these 5 patients
df_duck_all_records = duck_con.execute(f"""
    SELECT prontuario, patient_PatientID, patient_PatientIDx, patient_FirstName as first_name, patient_unit_huntington, embryo_EmbryoID, embryo_EmbryoDate
    FROM gold.embryoscope_embrioes
    WHERE CAST(patient_PatientIDx AS VARCHAR) IN ({','.join("'" + px + "'" for px in sample_patient_idxs)})
""").df()
df_duck_all_records['patient_PatientIDx'] = df_duck_all_records['patient_PatientIDx'].astype(str)

ath_cur.execute(f"""
    SELECT prontuario, patient_id, patient_id_x, first_name, unit_huntington, embryo_embryo_id, embryo_embryo_date
    FROM gold_huntington_prod.embryoscope_embrioes
    WHERE patient_id_x IN ({','.join("'" + px + "'" for px in sample_patient_idxs)})
""")
df_ath_all_records = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
df_ath_all_records['patient_id_x'] = df_ath_all_records['patient_id_x'].astype(str)

# 4. Display the first record for each patient in each database (sorted by patient_id_x)
df_duck_first = df_duck_all_records.drop_duplicates(subset=['patient_PatientIDx']).sort_values(by='patient_PatientIDx')
df_ath_first = df_ath_all_records.drop_duplicates(subset=['patient_id_x']).sort_values(by='patient_id_x')

print("\n🔹 [DuckDB] First line for each of the 5 discrepant patients (Sorted):")
display(df_duck_first)

print("\n🔸 [Athena Prod] First line for each of the 5 discrepant patients (Sorted):")
display(df_ath_first)

# 5. Query spouse names from Athena silver_clinisys_prod.view_pacientes using DuckDB's valid prontuarios
print("\n🌸 [Athena silver_clinisys_prod] Wife & Husband names matching DuckDB prontuarios:")
sample_prontuarios = df_duck_first['prontuario'].dropna().tolist()
sample_prontuarios_clean = [int(p) for p in sample_prontuarios if str(p) != '-1' and pd.notnull(p)]

if sample_prontuarios_clean:
    pront_list_str = ','.join(str(p) for p in sample_prontuarios_clean)
    clinisys_q = f"""
        SELECT codigo as prontuario, esposa_nome, marido_nome 
        FROM silver_clinisys_prod.view_pacientes 
        WHERE codigo IN ({pront_list_str})
    """
    ath_cur.execute(clinisys_q)
    df_clinisys_names = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
    df_clinisys_names['prontuario'] = df_clinisys_names['prontuario'].astype(str)
    display(df_clinisys_names)
else:
    print("No valid prontuario values available to query Clinisys spouse names.")

Discrepant Patient IDxs identified: ['PC10T4L7_44613.4678133796', 'PC10T4L7_44584.5066074074', 'PC10T4L7_44551.3003739352', 'PC10T4L7_44540.4750945833', 'PC10T4L7_44539.4780777199']

🔹 [DuckDB] First line for each of the 5 discrepant patients (Sorted):


,prontuario,patient_PatientID,patient_PatientIDx,first_name,patient_unit_huntington,embryo_EmbryoID,embryo_EmbryoDate
49,164377,66299,PC10T4L7_44539.4780777199,"ARAUJO, ELIANE R.",Vila Mariana,D2021.12.09_S01366_I3253_P-5,2021-12-09
42,185335,87297,PC10T4L7_44540.4750945833,"KONISHI, DIONELLA B",Vila Mariana,D2021.12.10_S01370_I3253_P-6,2021-12-10
25,186579,186579,PC10T4L7_44551.3003739352,"SILVA, ALINE S.",Vila Mariana,D2021.12.21_S01404_I3253_P-2,2021-12-21
17,137074,38610,PC10T4L7_44584.5066074074,"FIORETTO, ITACIRA N. S. DA S.",Vila Mariana,D2022.01.23_S01436_I3253_P-7,2022-01-23
0,183580,183580,PC10T4L7_44613.4678133796,"NASCIMENTO, ROBERTA",Vila Mariana,D2022.02.21_S01501_I3253_P-4,2022-02-21



🔸 [Athena Prod] First line for each of the 5 discrepant patients (Sorted):


,prontuario,patient_id,patient_id_x,first_name,unit_huntington,embryo_embryo_id,embryo_embryo_date
0,-1,66.299,PC10T4L7_44539.4780777199,"ARAUJO, ELIANE R.",Vila Mariana,D2021.12.09_S01366_I3253_P-7,2021-12-09
7,-1,87.297,PC10T4L7_44540.4750945833,"KONISHI, DIONELLA B",Vila Mariana,D2021.12.10_S01370_I3253_P-7,2021-12-10
39,-1,186.579,PC10T4L7_44551.3003739352,"SILVA, ALINE S.",Vila Mariana,D2023.04.05_S02155_I3253_P-9,2023-04-05
14,-1,38.610,PC10T4L7_44584.5066074074,"FIORETTO, ITACIRA N. S. DA S.",Vila Mariana,D2022.01.23_S01436_I3253_P-8,2022-01-23
22,-1,183.580,PC10T4L7_44613.4678133796,"NASCIMENTO, ROBERTA",Vila Mariana,D2023.02.09_S02049_I3253_P-12,2023-02-09



🌸 [Athena silver_clinisys_prod] Wife & Husband names matching DuckDB prontuarios:


,prontuario,esposa_nome,marido_nome
0,137074,Itacira Nice Soares da Silva Fioretto,Cássio Leandro Pieretti
1,164377,Eliane Rocha de Araujo,Reynaldo Tadeu de Andrade
2,183580,Roberta Munarin do Nascimento,Adriano Savaris
3,185335,Dionelia Bonfim Konishi,Renato Konishi Yokoo
4,186579,Aline Soares Silva,José Rodrigo Eduardo da Silva


## 📑 Part 5: Discrepancy & Exclusive Row Analysis
This section calculates the total count of discrepant embryo records (`embryo_EmbryoID` / `embryo_embryo_id`) that exist exclusively in one database but not the other, applying the sync lag and server outage filters. It displays the total discrepancy volumes and a sample of the exclusive records.

In [21]:
# 1. Fetch all records from both databases with the reporting column set
print("Fetching all records from DuckDB...")
df_duck_all = duck_con.execute("""
    SELECT prontuario, patient_PatientID, patient_PatientIDx, patient_FirstName as first_name, patient_unit_huntington, embryo_EmbryoID, embryo_EmbryoDate
    FROM gold.embryoscope_embrioes
""").df()
df_duck_all['embryo_EmbryoID'] = df_duck_all['embryo_EmbryoID'].astype(str)

print("Fetching all records from AWS Athena Prod...")
ath_cur.execute("""
    SELECT prontuario, patient_id, patient_id_x, first_name, unit_huntington, embryo_embryo_id, embryo_embryo_date
    FROM gold_huntington_prod.embryoscope_embrioes
""")
df_ath_all = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
df_ath_all['embryo_embryo_id'] = df_ath_all['embryo_embryo_id'].astype(str)

# Get max dates for sync lag verification
max_ath_date = pd.to_datetime(df_ath_all['embryo_embryo_date']).max()
max_duck_date = pd.to_datetime(df_duck_all['embryo_EmbryoDate']).max()

# Convert to datetime for filtering
df_duck_all['datetime_EmbryoDate'] = pd.to_datetime(df_duck_all['embryo_EmbryoDate'])
df_ath_all['datetime_embryo_date'] = pd.to_datetime(df_ath_all['embryo_embryo_date'])

# Print active filters
print("\n⚠️ Discrepancy comparison filters applied:")
print("  - Ignored Ibirapuera records from 2025 (known server issue)")
print("  - Filtered out records at or after the max embryo date of the target database (sync lag buffer)")
print(f"  - DuckDB max embryo date: {max_duck_date.strftime('%Y-%m-%d') if pd.notnull(max_duck_date) else 'N/A'}")
print(f"  - AWS Athena max embryo date: {max_ath_date.strftime('%Y-%m-%d') if pd.notnull(max_ath_date) else 'N/A'}")

# 2. Find records ONLY in DuckDB (missing in Athena)
df_only_duck = df_duck_all[~df_duck_all['embryo_EmbryoID'].isin(df_ath_all['embryo_embryo_id'])].copy()
df_only_duck = df_only_duck[~(
    (df_only_duck['patient_unit_huntington'] == 'Ibirapuera') &
    (df_only_duck['datetime_EmbryoDate'].dt.year == 2025)
)]
df_only_duck = df_only_duck[df_only_duck['datetime_EmbryoDate'] < max_ath_date]

print(f"\n🔴 Total records only in DuckDB: {len(df_only_duck)}")
if not df_only_duck.empty:
    print("Sample of records only in DuckDB (Filtered):")
    display(df_only_duck.head(5))
else:
    print("No exclusive DuckDB records found matching the restrictions.")

# 3. Find records ONLY in AWS Athena Prod (missing in DuckDB)
df_only_ath = df_ath_all[~df_ath_all['embryo_embryo_id'].isin(df_duck_all['embryo_EmbryoID'])].copy()
df_only_ath = df_only_ath[~(
    (df_only_ath['unit_huntington'] == 'Ibirapuera') &
    (df_only_ath['datetime_embryo_date'].dt.year == 2025)
)]
df_only_ath = df_only_ath[df_only_ath['datetime_embryo_date'] < max_duck_date]

print(f"\n🔵 Total records only in AWS Athena Prod: {len(df_only_ath)}")
if not df_only_ath.empty:
    print("Sample of records only in AWS Athena Prod (Filtered):")
    display(df_only_ath.head(5))
else:
    print("No exclusive Athena records found matching the restrictions.")

Fetching all records from DuckDB...
Fetching all records from AWS Athena Prod...

⚠️ Discrepancy comparison filters applied:
  - Ignored Ibirapuera records from 2025 (known server issue)
  - Filtered out records at or after the max embryo date of the target database (sync lag buffer)
  - DuckDB max embryo date: 2026-07-20
  - AWS Athena max embryo date: 2026-07-19

🔴 Total records only in DuckDB: 0
No exclusive DuckDB records found matching the restrictions.

🔵 Total records only in AWS Athena Prod: 2974
Sample of records only in AWS Athena Prod (Filtered):


,prontuario,patient_id,patient_id_x,first_name,unit_huntington,embryo_embryo_id,embryo_embryo_date,datetime_embryo_date
377,-1,overnight test,NEXTGEN_43895.6049278935,overnight,Brasilia,D2020.03.05_S00007_I4120_P-16,2020-03-05,2020-03-05
378,-1,overnight test,NEXTGEN_43895.6049278935,overnight,Brasilia,D2020.03.05_S00007_I4120_P-15,2020-03-05,2020-03-05
379,-1,overnight test,NEXTGEN_43895.6049278935,overnight,Brasilia,D2020.03.05_S00007_I4120_P-14,2020-03-05,2020-03-05
380,-1,overnight test,NEXTGEN_43895.6049278935,overnight,Brasilia,D2020.03.05_S00007_I4120_P-13,2020-03-05,2020-03-05
381,-1,overnight test,NEXTGEN_43895.6049278935,overnight,Brasilia,D2020.03.05_S00007_I4120_P-12,2020-03-05,2020-03-05


In [24]:
df_only_ath

,prontuario,patient_id,patient_id_x,first_name,unit_huntington,embryo_embryo_id,embryo_embryo_date,datetime_embryo_date
377,-1,overnight test,NEXTGEN_43895.6049278935,overnight,Brasilia,D2020.03.05_S00007_I4120_P-16,2020-03-05,2020-03-05
378,-1,overnight test,NEXTGEN_43895.6049278935,overnight,Brasilia,D2020.03.05_S00007_I4120_P-15,2020-03-05,2020-03-05
379,-1,overnight test,NEXTGEN_43895.6049278935,overnight,Brasilia,D2020.03.05_S00007_I4120_P-14,2020-03-05,2020-03-05
380,-1,overnight test,NEXTGEN_43895.6049278935,overnight,Brasilia,D2020.03.05_S00007_I4120_P-13,2020-03-05,2020-03-05
381,-1,overnight test,NEXTGEN_43895.6049278935,overnight,Brasilia,D2020.03.05_S00007_I4120_P-12,2020-03-05,2020-03-05
...,...,...,...,...,...,...,...,...
142890,-1,847.334,PC1P7BHG_45560.4037189352,"MONTEIRO, ANA L. J. C.",Ibirapuera,D2026.05.01_S04929_I3166_P-5,2026-05-01,2026-05-01
142891,-1,847.334,PC1P7BHG_45560.4037189352,"MONTEIRO, ANA L. J. C.",Ibirapuera,D2026.05.01_S04929_I3166_P-4,2026-05-01,2026-05-01
142892,-1,847.334,PC1P7BHG_45560.4037189352,"MONTEIRO, ANA L. J. C.",Ibirapuera,D2026.05.01_S04929_I3166_P-3,2026-05-01,2026-05-01
142893,-1,847.334,PC1P7BHG_45560.4037189352,"MONTEIRO, ANA L. J. C.",Ibirapuera,D2026.05.01_S04929_I3166_P-2,2026-05-01,2026-05-01


## 🥚 Part 6: Embryo Number Mismatch Analysis
This section identifies embryo records that exist in **both** databases (matched by embryo ID) but have **different** values for the embryo number field (`embryo_embryo_number` in DuckDB vs. `embryo_number` in AWS Athena).

In [22]:
# 1. Fetch reporting column set plus embryo numbers from both databases
print("Fetching embryo records with embryo numbers from DuckDB...")
df_duck_num = duck_con.execute("""
    SELECT prontuario, patient_PatientID, patient_PatientIDx, patient_FirstName as first_name, patient_unit_huntington, embryo_EmbryoID, embryo_EmbryoDate, embryo_embryo_number
    FROM gold.embryoscope_embrioes
""").df()
df_duck_num['embryo_EmbryoID'] = df_duck_num['embryo_EmbryoID'].astype(str)

print("Fetching embryo records with embryo numbers from AWS Athena Prod...")
ath_cur.execute("""
    SELECT prontuario, patient_id, patient_id_x, first_name, unit_huntington, embryo_embryo_id, embryo_embryo_date, embryo_number
    FROM gold_huntington_prod.embryoscope_embrioes
""")
df_ath_num = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
df_ath_num['embryo_embryo_id'] = df_ath_num['embryo_embryo_id'].astype(str)

# 2. Join the databases on embryo ID (inner join to find records existing in both)
df_merged_num = pd.merge(
    df_duck_num, 
    df_ath_num, 
    left_on='embryo_EmbryoID', 
    right_on='embryo_embryo_id', 
    suffixes=('_duck', '_ath')
)

# 3. Clean and convert embryo numbers to numeric to ensure clean comparison
df_merged_num['embryo_num_clean_duck'] = pd.to_numeric(df_merged_num['embryo_embryo_number'], errors='coerce')
df_merged_num['embryo_num_clean_ath'] = pd.to_numeric(df_merged_num['embryo_number'], errors='coerce')

# 4. Find records where clean embryo numbers do not match
df_mismatched_num = df_merged_num[
    df_merged_num['embryo_num_clean_duck'] != df_merged_num['embryo_num_clean_ath']
]

print(f"\n⚠️ Total matched embryo records with differing embryo numbers: {len(df_mismatched_num)}")
if not df_mismatched_num.empty:
    print("Sample of mismatched embryo number records:")
    # Display clean selection showing comparative columns
    display_cols = [
        'embryo_EmbryoID', 'patient_PatientIDx', 'first_name_duck', 
        'embryo_embryo_number', 'embryo_number', 
        'patient_unit_huntington', 'unit_huntington'
    ]
    display(df_mismatched_num[display_cols].head(10))
else:
    print("No embryo number discrepancies found for overlapping embryo records.")

Fetching embryo records with embryo numbers from DuckDB...
Fetching embryo records with embryo numbers from AWS Athena Prod...

⚠️ Total matched embryo records with differing embryo numbers: 0
No embryo number discrepancies found for overlapping embryo records.


In [23]:
# Close database connections
print("Closing database connections...")
try:
    duck_con.close()
    print("DuckDB connection closed.")
except Exception as e:
    print(f"Error closing DuckDB connection: {e}")

try:
    ath_con.close()
    print("Athena connection closed.")
except Exception as e:
    print(f"Error closing Athena connection: {e}")

Closing database connections...
DuckDB connection closed.
Athena connection closed.
